# 02 · Train the disease classifier (fp16 mixed precision)

Kaggle T4 gives ~2.5 h for 12 epochs on the full 70k-image training set.
Two-stage schedule: 1 epoch head-only, then unfreeze everything at a lower LR.
The number to beat: **val macro-F1**, not accuracy — the classes are imbalanced.


In [1]:
# --- Universal Environment Setup & Git Pull ---
import sys
import os
import subprocess

REPO_NAME = "SKIT-DS-2023-2027-015"
REPO_URL = f"https://github.com/shankerjoshi/{REPO_NAME}.git"

if 'google.colab' in sys.modules:
    # Google Colab Setup
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    !git pull origin main
    !pip install -r requirements.txt -q
    !pip install -q onnx onnxruntime onnxconverter-common shap
    print("Colab ready at:", os.getcwd())

elif os.path.exists('/kaggle'):
    # Kaggle Setup
    REPO_DIR = f"/kaggle/working/{REPO_NAME}"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    subprocess.run(["git", "pull", "origin", "main"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "shap"], check=True)
    print("Kaggle ready at:", os.getcwd())

else:
    if os.path.abspath("..") not in sys.path:
        sys.path.insert(0, os.path.abspath(".."))

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nPyTorch: {torch.__version__} | Accelerator: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'})")
if not torch.cuda.is_available():
    print("⚠️ WARNING: GPU is not detected! Make sure your notebook Accelerator is set to GPU (T4) before training.")

Cloning into 'SKIT-DS-2023-2027-015'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 72 (delta 21), reused 61 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 56.09 KiB | 6.23 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/SKIT-DS-2023-2027-015
From https://github.com/shankerjoshi/SKIT-DS-2023-2027-015
 * branch            main       -> FETCH_HEAD
Already up to date.
Colab ready at: /content/SKIT-DS-2023-2027-015

PyTorch: 2.14.1+cpu | Accelerator: cpu (No GPU)
⚠️ WARNING: GPU is not detected! Make sure your notebook Accelerator is set to GPU (T4) before training.


In [4]:
# --- Prepare Clean Dataset (With Auto-Download for Colab) ---
import json
import os
import shutil
from pathlib import Path

MANIFEST_FILE = Path("data/leakage_drop_manifest.json")
FAST_TRAIN = Path("data/fast/plantvillage/train")
FAST_VALID = Path("data/fast/plantvillage/valid")
RAW_PV = Path("data/raw/plantvillage")

FAST_TRAIN.mkdir(parents=True, exist_ok=True)
FAST_VALID.mkdir(parents=True, exist_ok=True)
RAW_PV.mkdir(parents=True, exist_ok=True)

# 1. Load the drop manifest from Notebook 01
drop_train_files = set()
drop_valid_files = set()

if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE) as f:
        manifest = json.load(f)
    drop_train_files = set(os.path.basename(p) for p in manifest.get("drop_from_train_paths", []))
    drop_valid_files = set(os.path.basename(p) for p in manifest.get("drop_from_valid_paths", []))
    print(f"✓ Loaded drop manifest: filtering {len(drop_train_files)} train images and {len(drop_valid_files)} valid images.")
else:
    print("Notice: data/leakage_drop_manifest.json not found. Proceeding with all images.")

# 2. Check if raw dataset exists; if not, download via kagglehub
def find_split_dir(split_name="train"):
    candidates = [
        RAW_PV / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)" / split_name,
        RAW_PV / "new plant diseases dataset(augmented)" / "New Plant Diseases Dataset(Augmented)" / split_name,
        RAW_PV / split_name,
        Path(f"data/raw/{split_name}")
    ]
    for c in candidates:
        if c.exists() and any(c.iterdir()):
            return c
    return None

raw_train = find_split_dir("train")
raw_valid = find_split_dir("valid")

if raw_train is None or raw_valid is None:
    print("\nRaw dataset not found on Colab disk. Downloading via kagglehub...")
    !pip install -q kagglehub

    # Retrieve Kaggle credentials if stored in Colab Secrets
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
            os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
        except Exception:
            pass

    import kagglehub
    download_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
    print(f"Downloaded to cache: {download_path}")

    # Link or copy into data/raw/plantvillage
    for item in Path(download_path).iterdir():
        target = RAW_PV / item.name
        if not target.exists():
            os.symlink(item, target)

    raw_train = find_split_dir("train")
    raw_valid = find_split_dir("valid")

print(f"\n✓ Located source train: {raw_train}")
print(f"✓ Located source valid: {raw_valid}")

# 3. Populate data/fast with filtered clean images
def link_clean_images(src_dir, dest_dir, drop_filenames):
    total = 0
    skipped = 0
    for root, _, files in os.walk(src_dir, followlinks=True):
        class_name = os.path.basename(root)
        if class_name.lower() in ["train", "valid"]:
            continue
        dest_class_dir = dest_dir / class_name
        dest_class_dir.mkdir(parents=True, exist_ok=True)

        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                if f in drop_filenames:
                    skipped += 1
                    continue
                src_path = os.path.join(root, f)
                target_path = dest_class_dir / f
                if not target_path.exists():
                    os.symlink(src_path, target_path)
                total += 1
    return total, skipped

print("\nPreparing clean train set...")
clean_train_count, skipped_train = link_clean_images(raw_train, FAST_TRAIN, drop_train_files)
print(f"✓ Clean training images ready: {clean_train_count:,} (excluded {skipped_train} leaked/duplicates)")

print("Preparing clean validation set...")
clean_valid_count, skipped_valid = link_clean_images(raw_valid, FAST_VALID, drop_valid_files)
print(f"✓ Clean validation images ready: {clean_valid_count:,} (excluded {skipped_valid} leaked/duplicates)")

✓ Loaded drop manifest: filtering 24 train images and 35 valid images.

Raw dataset not found on Colab disk. Downloading via kagglehub...
Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Downloaded to cache: /kaggle/input/new-plant-diseases-dataset

✓ Located source train: data/raw/plantvillage/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
✓ Located source valid: data/raw/plantvillage/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid

Preparing clean train set...
✓ Clean training images ready: 70,271 (excluded 24 leaked/duplicates)
Preparing clean validation set...
✓ Clean validation images ready: 17,537 (excluded 35 leaked/duplicates)


In [6]:
# --- Recreate Clean Fast Dataset With Absolute Symlinks ---
import json
import os
import shutil
from pathlib import Path

FAST_TRAIN = Path("data/fast/plantvillage/train")
FAST_VALID = Path("data/fast/plantvillage/valid")
RAW_PV = Path("data/raw/plantvillage")
MANIFEST_FILE = Path("data/leakage_drop_manifest.json")

# 1. Clean up existing broken symlink directories
if FAST_TRAIN.exists():
    shutil.rmtree(FAST_TRAIN)
if FAST_VALID.exists():
    shutil.rmtree(FAST_VALID)

FAST_TRAIN.mkdir(parents=True, exist_ok=True)
FAST_VALID.mkdir(parents=True, exist_ok=True)

# 2. Load the drop manifest
drop_train_files = set()
drop_valid_files = set()
if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE) as f:
        manifest = json.load(f)
    drop_train_files = set(os.path.basename(p) for p in manifest.get("drop_from_train_paths", []))
    drop_valid_files = set(os.path.basename(p) for p in manifest.get("drop_from_valid_paths", []))

# 3. Locate source raw directories
def find_split_dir(split_name="train"):
    candidates = [
        RAW_PV / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)" / split_name,
        RAW_PV / "new plant diseases dataset(augmented)" / "New Plant Diseases Dataset(Augmented)" / split_name,
        RAW_PV / split_name,
        Path(f"data/raw/{split_name}")
    ]
    for c in candidates:
        if c.exists() and any(c.iterdir()):
            return c
    return None

raw_train = find_split_dir("train")
raw_valid = find_split_dir("valid")

# 4. Create symlinks using ABSOLUTE paths
def link_clean_images_abs(src_dir, dest_dir, drop_filenames):
    total = 0
    skipped = 0
    for root, _, files in os.walk(src_dir, followlinks=True):
        class_name = os.path.basename(root)
        if class_name.lower() in ["train", "valid"]:
            continue
        dest_class_dir = dest_dir / class_name
        dest_class_dir.mkdir(parents=True, exist_ok=True)

        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                if f in drop_filenames:
                    skipped += 1
                    continue
                # Use abspath so the symlink resolves from any directory
                src_abs = os.path.abspath(os.path.join(root, f))
                target_path = dest_class_dir / f
                if not target_path.exists():
                    os.symlink(src_abs, target_path)
                total += 1
    return total, skipped

print("Rebuilding clean train dataset with absolute symlinks...")
train_count, s_train = link_clean_images_abs(raw_train, FAST_TRAIN, drop_train_files)
print(f"✓ Clean train images: {train_count:,} (excluded {s_train})")

print("Rebuilding clean valid dataset with absolute symlinks...")
valid_count, s_valid = link_clean_images_abs(raw_valid, FAST_VALID, drop_valid_files)
print(f"✓ Clean valid images: {valid_count:,} (excluded {s_valid})")

# 5. Quick verification test (ensures PIL can read the symlinked files)
sample_file = next(FAST_TRAIN.rglob("*.JPG"), next(FAST_TRAIN.rglob("*.jpg"), None))
if sample_file:
    from PIL import Image
    with Image.open(sample_file) as img:
        print(f"✓ File reading verified! Sample image size: {img.size}")

Rebuilding clean train dataset with absolute symlinks...
✓ Clean train images: 70,271 (excluded 24)
Rebuilding clean valid dataset with absolute symlinks...
✓ Clean valid images: 17,537 (excluded 35)
✓ File reading verified! Sample image size: (256, 256)


### Find the CPU sweet spot first
Each DataLoader worker is a separate process decoding JPEGs. Too few starve the GPU; more than the vCPU count oversubscribes and everything slows down. Measure it once, then use the winner.


In [7]:
import os, time
from torch.utils.data import DataLoader
from src.data.dataset import LeafDataset, build_transforms, index_folder

TRAIN_DIR = "data/fast/plantvillage/train"   # from data/prepare_fast.py; falls back to the raw dir
items, classes = index_folder(TRAIN_DIR)
ds = LeafDataset(items[:4000], build_transforms(True, 224), classes)
print("vCPUs available:", os.cpu_count())

for w in [0, 1, 2, 4]:
    kw = {"prefetch_factor": 4} if w else {}
    dl = DataLoader(ds, batch_size=64, num_workers=w, pin_memory=True, persistent_workers=w > 0, **kw)
    it = iter(dl); next(it)                       # warm-up batch
    t0 = time.perf_counter(); n = 0
    for i, _ in enumerate(it):
        n += 64
        if i == 19:
            break
    print(f"workers={w}: {n / (time.perf_counter() - t0):6.1f} images/s")


vCPUs available: 2
workers=0:   50.6 images/s


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


workers=1:   79.3 images/s
workers=2:  111.3 images/s


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


workers=4:  106.9 images/s


Use the best worker count below, and try `--batch-size 128` once — if it does not raise CUDA out-of-memory, it is free throughput.


In [ ]:
!python -m src.train --config configs/train_disease.yaml --batch-size 128


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
h = pd.DataFrame(json.load(open("runs/history.json")))
display(h)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_macro_f1, label="val macro-F1")
ax[1].plot(h.epoch, h.val_acc, label="val accuracy"); ax[1].legend(); ax[1].set_title("metrics")
plt.tight_layout(); plt.savefig("results_training_curve.png", dpi=110); plt.show()
print("best macro-F1:", h.val_macro_f1.max())


### Before you close this notebook
- Save `runs/best.pt` to the Kaggle output (or push it to a GitHub Release) — it is gitignored on purpose.
- Record the epoch count, batch size, LR and the best macro-F1 in the report table.
